In [ ]:
# Problema: Organizar datos históricos en particiones Parquet locales y comprobar que puedan recuperarse correctamente.

"""Organiza una historia de ventas en particiones locales de Parquet."""
from pathlib import Path
import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
SOURCE, LAKE, OUTPUT = ROOT / "data/sales_history.parquet", ROOT / "temp/lake/curated/sales", ROOT / "submission/lake_summary.csv"


def build_submission():
    rows = [{"transaction_id": f"T6{i:03d}", "transaction_date": f"{year}-{month:02d}-{(i % 20) + 1:02d}", "customer_id": f"C{i % 6:03d}", "product_id": f"P{i % 4:03d}", "quantity": (i % 4) + 1, "sales_amount": float((i % 4 + 1) * 10)} for i, (year, month) in enumerate([(2025,11),(2025,12),(2026,1),(2026,2)] * 10)]
    frame = pd.DataFrame(rows); frame.to_parquet(SOURCE, index=False, engine="pyarrow")
    if LAKE.exists():
        for file in LAKE.rglob("*.parquet"): file.unlink()
    frame["year"] = frame.transaction_date.str[:4].astype(int); frame["month"] = frame.transaction_date.str[5:7].astype(int)
    for (year, month), partition in frame.groupby(["year", "month"]):
        path = LAKE / f"year={year}" / f"month={month:02d}"; path.mkdir(parents=True, exist_ok=True)
        partition.drop(columns=["year", "month"]).to_parquet(path / "sales.parquet", index=False, engine="pyarrow")
    files = list(LAKE.rglob("*.parquet")); restored = pd.concat([pd.read_parquet(file, engine="pyarrow") for file in files])
    assert len(restored) == len(frame)
    pd.DataFrame([["sales", "year,month", 4, len(files), len(frame)]], columns=["dataset","partition_columns","partition_count","file_count","row_count"]).to_csv(OUTPUT, index=False)


if __name__ == "__main__": build_submission()